# Multi-Seed EgyBERT Comparison

Runs the matched `L1+L2+L3` `EgyBERT` comparison across three fixed seeds: `42`, `123`, and `456`.

This notebook mirrors the structure of `kaggle_multi_seed_training.ipynb`, but limits the run set to the single `EgyBERT` comparison configuration.

In [1]:
# Clone the repository
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding the cloud's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml -q

Cloning into 'arabic-itsm-classification'...
remote: Enumerating objects: 275, done.
remote: Counting objects: 100% (275/275), done.
remote: Compressing objects: 100% (186/186), done.
remote: Total 275 (delta 150), reused 197 (delta 81), pack-reused 0 (from 0)
Receiving objects: 100% (275/275), 1.67 MiB | 13.69 MiB/s, done.
Resolving deltas: 100% (150/150), done.
/kaggle/working/arabic-itsm-classification
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 80.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 54.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.

In [2]:
# Link the processed dataset
!rm -rf data/processed && mkdir -p data/processed
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/
!ls data/processed
# Expected: label_encoders.pkl  test.csv  train.csv  val.csv

label_encoders.pkl  test.csv  train.csv  val.csv


In [3]:
!nvidia-smi

Sun Apr  5 18:28:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# Download EgyBERT fully to local disk before training.
from huggingface_hub import snapshot_download

print('Downloading EgyBERT to local disk...')
EGYBERT_PATH = snapshot_download(
    'faisalq/EgyBERT',
    local_dir='/kaggle/working/model_cache/egybert'
)
print('EgyBERT ready at:', EGYBERT_PATH)

Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]

EgyBERT ready at: /kaggle/working/model_cache/egybert


In [5]:
from pathlib import Path

METRICS_DIR = Path('results/metrics/multi_seed')
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]

CONFIGS = [
    {
        'name': 'egybert_l1l2l3',
        'model': EGYBERT_PATH,
        'flags': '--tasks l1 l2 l3',
        'task_label': 'l1_l2_l3',
        'checkpoint_prefix': 'egybert',
        'epochs': 10,
    }
]

print('Total runs:', len(CONFIGS) * len(SEEDS))
for cfg in CONFIGS:
    for seed in SEEDS:
        print(' ', cfg['name'], 'seed=' + str(seed))

Total runs: 3
  egybert_l1l2l3 seed=42
  egybert_l1l2l3 seed=123
  egybert_l1l2l3 seed=456


In [6]:
import json, time

ipy = get_ipython()
run_log = []

for cfg in CONFIGS:
    for seed in SEEDS:
        run_key = cfg['name'] + '_seed' + str(seed)
        out_metrics_path = METRICS_DIR / (run_key + '_metrics.json')

        if out_metrics_path.exists():
            print('[SKIP]', run_key, '-- already exists')
            continue

        seed_dir = 'models/seed_' + str(seed)
        checkpoint_dir = Path(seed_dir) / (cfg['checkpoint_prefix'] + '_' + cfg['task_label'] + '_best')

        parts = [
            'python scripts/train.py',
            cfg['flags'],
            '--model', cfg['model'],
            '--epochs', str(cfg['epochs']),
            '--seed', str(seed),
            '--output-dir', seed_dir,
            '--data-dir data/processed',
        ]
        cmd = ' '.join(parts)

        print('')
        print('[RUN]', run_key)
        print('  CMD:', cmd)
        print('  Checkpoint will be:', checkpoint_dir)
        t0 = time.time()
        ipy.system(cmd)
        elapsed = time.time() - t0
        print('  Time: %.1f min' % (elapsed / 60,))

        test_metrics_file = checkpoint_dir / 'test_metrics.json'
        if test_metrics_file.exists():
            with open(test_metrics_file) as fp:
                test_metrics = json.load(fp)
            result = {'config': cfg['name'], 'seed': seed, 'metrics': test_metrics}
            with open(out_metrics_path, 'w') as fp:
                json.dump(result, fp, indent=2)
            print('  Saved:', out_metrics_path)
            run_log.append({'run': run_key, 'status': 'ok', 'time_min': elapsed / 60})
        else:
            print('  WARNING: test_metrics.json not found at', test_metrics_file)
            run_log.append({'run': run_key, 'status': 'no_metrics', 'time_min': elapsed / 60})

print('')
print('=== Run log ===')
for r in run_log:
    print(' ', r['run'] + ':', r['status'], '(%.1f min)' % r['time_min'])


[RUN] egybert_l1l2l3_seed42
  CMD: python scripts/train.py --tasks l1 l2 l3 --model /kaggle/working/model_cache/egybert --epochs 10 --seed 42 --output-dir models/seed_42 --data-dir data/processed
  Checkpoint will be: models/seed_42/egybert_l1_l2_l3_best
Device: cuda | FP16: True
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
Tasks: ['l1', 'l2', 'l3'] | Classes: {'l1': 6, 'l2': 16, 'l3': 48}
Loading weights: 100%|█| 199/199 [00:00<00:00, 1684.34it/s, Materializing param=
2026/04/05 18:28:45 INFO mlflow.tracking.fluent: Experiment with name 'arabic-itsm-transformers' does not exist. Creating a new experiment.
Epoch 1: 100%|█████████████████████████

In [7]:
# Aggregate results
!python scripts/aggregate_multi_seed.py --metrics-dir results/metrics/multi_seed

Loading results from results/metrics/multi_seed...

egybert_l1l2l3: 3 seeds
  l1: 0.8843 ± 0.0009  (values: [0.883, 0.885, 0.8848])
  l2: 0.8685 ± 0.0045  (values: [0.8719, 0.8715, 0.8621])
  l3: 0.2306 ± 0.0048  (values: [0.2286, 0.2259, 0.2371])


=== Paper-ready table (mean ± std ×100) ===
Config                              L1           L2           L3     Priority    Sentiment
-------------------------------------------------------------------------------------
egybert_l1l2l3              88.43±0.09   86.85±0.45   23.06±0.48            —            —

Saved to results/metrics/multi_seed_summary.json


In [8]:
# View only the EgyBERT summary row
import json
with open('results/metrics/multi_seed_summary.json') as f:
    summary = json.load(f)

data = summary['summary']['egybert_l1l2l3']
print('egybert_l1l2l3 (seeds: %s):' % data['seeds'])
for task, stats in data['tasks'].items():
    print(f'  {task}: {stats["mean"]*100:.2f} ± {stats["std"]*100:.2f} (F1%)')

egybert_l1l2l3 (seeds: [123, 42, 456]):
  l1: 88.43 ± 0.09 (F1%)
  l2: 86.85 ± 0.45 (F1%)
  l3: 23.06 ± 0.48 (F1%)


In [9]:
# Package the EgyBERT multi-seed metrics for download
!zip -r egybert_multi_seed_results.zip results/metrics/multi_seed/egybert_l1l2l3_seed*_metrics.json results/metrics/multi_seed_summary.json
print('Created egybert_multi_seed_results.zip - download from Kaggle output')

  adding: results/metrics/multi_seed/egybert_l1l2l3_seed123_metrics.json (deflated 44%)
  adding: results/metrics/multi_seed/egybert_l1l2l3_seed42_metrics.json (deflated 43%)
  adding: results/metrics/multi_seed/egybert_l1l2l3_seed456_metrics.json (deflated 44%)
  adding: results/metrics/multi_seed_summary.json (deflated 76%)
Created egybert_multi_seed_results.zip - download from Kaggle output
